In [1]:
import mlflow
from mlflow.tracking import MlflowClient


In [2]:
mlflow.set_tracking_uri("http://localhost:5001")

In [3]:
mlclient = MlflowClient()

In [4]:
run_id = "5c67fb931b5148c1aa0b1bcb15a4a328"

In [7]:
run = mlclient.get_run(run_id)
last_checkpoint_step = run.data.tags.get("last_checkpoint_step")
last_checkpoint_step

'5500'

In [8]:
latest_checkpoint_folder = f'checkpoint_{last_checkpoint_step}'
latest_checkpoint_folder

'checkpoint_5500'

In [9]:
model_path = mlclient.download_artifacts(run_id, f"{latest_checkpoint_folder}/artifacts/model.zip")

In [10]:
model_path

'C:\\Users\\coool\\AppData\\Local\\Temp\\tmpkcw4pft9\\checkpoint_5500\\artifacts/model.zip'

In [5]:
model_artifacts = mlclient.list_artifacts(run_id, path='model')   

In [6]:
model_artifacts

[]

In [9]:
from trading_functions.db.session import SessionLocal
from sqlalchemy.orm import Session

From inside trading_functions, user :  cooolrahul_postgres
Using Postgres DB: INF_DB at postgres:5432 with user cooolrahul_postgres


In [7]:
from rl_functions.utils import get_closest_trading_date

In [8]:
import datetime

In [9]:
start_date = datetime.datetime.strptime('2023-01-01', "%Y-%m-%d").date()
start_date

datetime.date(2023, 1, 1)

In [10]:
from dotenv import load_dotenv

In [11]:
load_dotenv("../.env_local", override=True)

True

In [12]:
import os
os.getenv("POSTGRES_HOST")

'localhost'

In [8]:
db: Session = SessionLocal()

NameError: name 'SessionLocal' is not defined

In [14]:
first_date = get_closest_trading_date(db=db, 
                         symbol='SPY',
                         target_date=start_date
                         )

In [19]:
first_date

(datetime.datetime(2023, 1, 3, 9, 30),)

In [22]:
second_date = get_closest_trading_date(db=db, 
                         symbol='SPY',
                         target_date=first_date.time.date() + datetime.timedelta(days=1),
                         only_next=True
                         )

In [23]:
second_date

(datetime.datetime(2023, 1, 4, 9, 30),)

### Testing the issue with get data starting from 11 AM

In [1]:
import yaml
import logging
import mlflow
from mlflow.tracking import MlflowClient
import datetime
from trading_functions.db.session import SessionLocal
from sqlalchemy.orm import Session
from dotenv import load_dotenv
import os
db: Session = SessionLocal()
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger(__name__)

From inside trading_functions, user :  cooolrahul_postgres
Using Postgres DB: INF_DB at localhost:5432 with user cooolrahul_postgres


In [2]:
def get_observation_features(inf_config: dict):
    slopes = inf_config['rl']['slopes']
    slope_column_numbers = [int(s) for s in slopes.split(',')]
    base_columns = ['pred_high', 'pred_low', 'pred_high_error', 'pred_low_error', 'pred_high_diff', 'pred_low_diff', 'momentum_short', 'velocity_short']
    for s in slope_column_numbers:
        base_columns.append(f'slope_last{s}close')
        base_columns.append(f'pred_high_slope_last{s}')
        base_columns.append(f'pred_low_slope_last{s}')
    return base_columns

In [3]:
def get_env(config: dict, db: Session, eval_mode: bool=False, dagster_logger=None):
    """
    Creates and returns a DummyVecEnv wrapped TradingEnv instance based on the provided configuration and database session.
    If eval_mode is True, it sets up the environment for evaluation using test data; otherwise, it sets up for training using training data.
    """
    from rl_functions.trading_env import TradingEnv

    obs_features = get_observation_features(config)
    start_date_config_key = 'train_start_date' if not eval_mode else 'test_start_date'
    end_date_config_key = 'train_end_date' if not eval_mode else 'test_end_date' 
    start_date = datetime.datetime.strptime(config['rl'][start_date_config_key], "%Y-%m-%d").date()
    if not eval_mode:
        start_date_first = get_closest_trading_date(db=db, symbol='SPY', target_date=start_date, only_next=True).time.date()
        start_date = get_closest_trading_date(db=db, symbol='SPY', target_date=start_date_first + datetime.timedelta(days=1), only_next=True).time.date()
    end_date = datetime.datetime.strptime(config['rl'][end_date_config_key], "%Y-%m-%d").date()
    initial_balance = float(config['rl']['initial_balance'])
    trade_fee = float(config['rl']['trade_fee'])
    max_trade_loss_percent = float(config['rl']['max_trade_loss_percent'])
    price_multiplier = float(config['rl']['price_multiplier'])


    def make_env():
        return TradingEnv(db=db,
                          symbol='SPY',
                          start_date=start_date,
                          end_date=end_date,
                          initial_balance=initial_balance,
                          trade_fee=trade_fee,
                          max_trade_loss_percent=max_trade_loss_percent,
                          obs_features=obs_features,
                          price_multiplier=price_multiplier,
                            evaluation=eval_mode,
                            logger=dagster_logger
                          )
    
    return make_env()

In [4]:
mlflow.set_tracking_uri("http://localhost:5001")

In [5]:
logger.info("Starting test_mlflow_download_model")

2026-02-20 23:33:04,366 [INFO] __main__: Starting test_mlflow_download_model


In [6]:
config_path = "../Config/config_dev.yaml"

In [7]:
with open(config_path, 'r') as file:
    conf = yaml.safe_load(file)
            

In [8]:
os.getenv("POSTGRES_HOST")

'localhost'

In [9]:

eval_env = get_env(
            config=conf,
            db=db,
            eval_mode=True,
            dagster_logger=logger)

2026-02-20 23:33:11,391 [INFO] __main__: Initializing TradingEnv for symbol: SPY, start_date: 2023-04-03, end_date: 2023-04-09, evaluation: True
[2026-02-20 23:33:11,391] INFO __main__: Initializing TradingEnv for symbol: SPY, start_date: 2023-04-03, end_date: 2023-04-09, evaluation: True
2026-02-20 23:33:11,458 [INFO] rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
[2026-02-20 23:33:11,458] INFO rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
2026-02-20 23:33:11,477 [INFO] rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
[2026-02-20 23:33:11,477] INFO rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
2026-02-20 23:33:11,494 [INFO] root: Using default MLflow tracking URI from config : http://localhost:5001
[2026-02-20 23:

Evaluation mode: ON


2026-02-20 23:33:11,689 [INFO] root: Downloading artifacts to /model_local_artifacts
[2026-02-20 23:33:11,689] INFO root: Downloading artifacts to /model_local_artifacts
c:\Users\coool\AppData\Local\pypoetry\Cache\virtualenvs\rahul-trading--q5kPROH-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


run_model/scalers.pkl  (dir: False)
run_model/training_config.yaml  (dir: False)


2026-02-20 23:33:11,781 [INFO] root: Scalers downloaded to c:\model_local_artifacts\scalers.pkl
[2026-02-20 23:33:11,781] INFO root: Scalers downloaded to c:\model_local_artifacts\scalers.pkl
2026-02-20 23:33:11,784 [INFO] root: Scalers loaded: dict_keys(['minmax', 'standard', 'robust'])
[2026-02-20 23:33:11,784] INFO root: Scalers loaded: dict_keys(['minmax', 'standard', 'robust'])
2026-02-20 23:33:11,951 [INFO] root: Config downloaded to c:\model_local_artifacts\training_config.yaml
[2026-02-20 23:33:11,951] INFO root: Config downloaded to c:\model_local_artifacts\training_config.yaml
2026/02/20 23:33:12 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/02/20 23:33:1

Previous date entry found:  2023-03-31 15:59:00
Querying previous day data from 2023-03-31 00:00:00 to 2023-04-01 00:00:00
Previous day data count: 390
1st entry: 2023-03-31 09:30:00, Last entry: 2023-03-31 15:59:00
Querying price data for SPY from 2023-04-03 00:00:00 to 2023-04-04 00:00:00


2026-02-20 23:33:16,164 [INFO] root: Data shape after adding indicators: (489, 74)
[2026-02-20 23:33:16,164] INFO root: Data shape after adding indicators: (489, 74)
2026-02-20 23:33:16,170 [INFO] root: Data shape before scaling: (489, 74)
[2026-02-20 23:33:16,170] INFO root: Data shape before scaling: (489, 74)


Data shape after transformation: (489, 74)
                   Date      Open      High     Low     Close  Volume  \
99  2023-03-31 14:21:00  406.8350  406.9300  406.81  406.8400  175823   
100 2023-03-31 14:22:00  406.8350  406.8400  406.66  406.7089  140276   
101 2023-03-31 14:23:00  406.7100  406.7400  406.63  406.7301  106459   
102 2023-03-31 14:24:00  406.7350  406.7400  406.64  406.6500   77004   
103 2023-03-31 14:25:00  406.6699  406.6699  406.58  406.6007   99317   

         MA20      MA50     MA100     EMA20  ...  fourier_imag_24  Volatility  \
99   0.277024 -0.167514 -0.011513  0.134801  ...         0.069706   -0.993566   
100  0.525034  0.004564  0.091234  0.412511  ...        -0.104718   -0.968109   
101  0.478217 -0.010747  0.074356  0.337632  ...         0.191896   -0.978354   
102  0.615782  0.095748  0.137354  0.486876  ...        -0.245019   -0.965016   
103  0.673788  0.164782  0.176696  0.555958  ...         0.327870   -0.992260   

     Momentum      PVPT    PVPT

2026-02-20 23:33:17,079 [INFO] __main__: Data shape : (390, 94)
[2026-02-20 23:33:17,079] INFO __main__: Data shape : (390, 94)
2026-02-20 23:33:17,085 [INFO] __main__: Data time range : 2023-04-03 09:30:00 to 2023-04-03 15:59:00
[2026-02-20 23:33:17,085] INFO __main__: Data time range : 2023-04-03 09:30:00 to 2023-04-03 15:59:00


Final df after removing prev rows
                 Date     Open     High      Low     Close  Volume      MA20  \
0 2023-04-03 09:30:00  408.850  409.030  408.820  408.9500  554637 -1.680463   
1 2023-04-03 09:31:00  408.940  409.240  408.930  409.2200  425640 -2.063047   
2 2023-04-03 09:32:00  409.225  409.240  409.060  409.1700  167566 -1.817903   
3 2023-04-03 09:33:00  409.160  409.385  409.100  409.3200  234395 -1.953591   
4 2023-04-03 09:34:00  409.340  409.490  409.335  409.4555  296118 -2.035750   

       MA50     MA100     EMA20  ...  pred_high_slope_last39  \
0 -0.959198 -0.970576 -1.455476  ...                0.002013   
1 -1.247411 -1.164241 -1.880716  ...                0.005584   
2 -1.142875 -1.105478 -1.585583  ...                0.008906   
3 -1.280871 -1.202985 -1.743345  ...                0.012805   
4 -1.397992 -1.287328 -1.855144  ...                0.017182   

   pred_high_slope_last99  pred_low_slope_last3  pred_low_slope_last9  \
0                0.015108  

In [14]:
eval_env.data.head()

,Date,Open,High,Low,Close,Volume,MA20,MA50,MA100,EMA20,...,pred_high_slope_last39,pred_high_slope_last99,pred_low_slope_last3,pred_low_slope_last9,pred_low_slope_last39,pred_low_slope_last99,pred_high_error,pred_low_error,momentum_short,velocity_short
0,2023-04-03 11:09:00,408.6300,408.760,408.59,408.7317,114828,0.821324,1.776435,1.278329,1.124340,...,-0.053014,-0.000083,0.000893,-0.021423,-0.100982,-0.014666,-0.219827,-1.181675,-0.001777,0.000045
1,2023-04-03 11:10:00,408.7300,408.730,408.63,408.6790,143583,0.833506,1.782688,1.318000,1.139496,...,-0.052591,-0.001516,0.204074,0.036376,-0.097665,-0.016181,-0.024078,-0.880100,-0.001729,0.000048
2,2023-04-03 11:11:00,408.6779,408.680,408.44,408.4800,144477,1.146531,1.965562,1.470135,1.465515,...,-0.051751,-0.003008,-0.028186,0.035773,-0.094655,-0.017861,-0.029019,-1.088047,-0.001756,-0.000027
3,2023-04-03 11:12:00,408.4850,408.755,408.44,408.7200,206651,0.612340,1.610070,1.276060,0.823651,...,-0.048853,-0.004107,-0.213060,0.038689,-0.091752,-0.019492,0.539882,-1.156220,-0.001652,0.000104
4,2023-04-03 11:13:00,408.7100,408.710,408.53,408.5500,116958,0.869123,1.753923,1.404894,1.117675,...,-0.046743,-0.005330,-0.003144,0.007722,-0.087631,-0.020968,0.221185,-1.094335,-0.001577,0.000075


In [3]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5001")
from mlflow.tracking import MlflowClient

def register_checkpoint_to_registry(run_id: str, checkpoint_folder: str, model_name: str):
    """
    Registers a specific checkpoint folder from a run to the Model Registry.
    
    Args:
        run_id: The ID of the MLflow run.
        checkpoint_folder: The folder name (e.g., 'checkpoint_1000').
        model_name: The name you want in the Model Registry.
    """
    client = MlflowClient()
    
    # 1. Construct the URI pointing to the specific checkpoint folder in the run
    # Format: runs:/<run_id>/<path_to_checkpoint>
    model_uri = f"runs:/{run_id}/{checkpoint_folder}"
    
    print(f"Registering model from: {model_uri}")
    
    # 2. Register the model
    # This creates the link between the Registry and the Run
    result = mlflow.register_model(model_uri, model_name)
    
    print(f"Successfully registered {model_name} version {result.version}")
    return result

In [4]:
run_id = "271dcc1c9d2d42ff8e39d895f1cb907c"
model_folder_path = "checkpoint_5500"
model_registry_name = "PPO_Trading_Model"
register_checkpoint_to_registry(run_id, model_folder_path, model_registry_name)



Registering model from: runs:/271dcc1c9d2d42ff8e39d895f1cb907c/checkpoint_5500


Successfully registered model 'PPO_Trading_Model'.
2026/02/24 19:48:51 WARNING mlflow.tracking._model_registry.fluent: Run with id 271dcc1c9d2d42ff8e39d895f1cb907c has no artifacts at artifact path 'checkpoint_5500', registering model based on models:/m-922467fd45764442bd91aa8ee8ae1523 instead
2026/02/24 19:48:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PPO_Trading_Model, version 1


Successfully registered PPO_Trading_Model version 1


Created version '1' of model 'PPO_Trading_Model'.


<ModelVersion: aliases=[], creation_timestamp=1771980531974, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1771980531974, metrics=None, model_id=None, name='PPO_Trading_Model', params=None, run_id='271dcc1c9d2d42ff8e39d895f1cb907c', run_link='', source='models:/m-922467fd45764442bd91aa8ee8ae1523', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [13]:

model_folder_path = "checkpoint_5500/artifacts"
version = 1
mlflow.set_tracking_uri("http://localhost:5001")
client = MlflowClient()
model_version_details = client.get_model_version(name=model_registry_name, version=version)
source_uri = model_version_details.source
run_id = model_version_details.run_id
print(f"Model version source URI: {source_uri}")
artifacts = client.list_artifacts(run_id=run_id, path=model_folder_path)
for art in artifacts:
    print(f"Found artifact: {art.path}")

Model version source URI: models:/m-922467fd45764442bd91aa8ee8ae1523
Found artifact: checkpoint_5500/artifacts/model.zip
Found artifact: checkpoint_5500/artifacts/vec_normalize.pkl
